 # 04_C3.py : C3, TF-C pre-training on the unlabelled pool (+ labelled TRAIN windows), then supervised fine-tune of the same dense V-Net

 Needs `units_C0` and `fold_splits.json` from 01_C0.py (for the fold splits and seeds). Z-score preprocessing, stitched evaluation.

 The TF-C time encoder is a full-width dense V-Net; its weights (without the classifier) initialise the identical V-Net used by C0,

 trained with the same recipe and early stopping. C3 gets extra compute (pre-training) that C0+ does not match.

 Output: `units_C3/fXX_sY.pkl` + `models/C3/*.pth`. Resumable.

In [ ]:
from thesis_common import *

POOLS = ["A"]                 # table in 05_evaluate uses pool "A"; add "B", "AB" for the pool factor (seed 0 only)
init_data(unlab=sorted(set("".join(POOLS))))

TFC_TAU, TFC_LAM, TFC_DELTA, SEG = 0.2, 0.5, 1.0, 25
NSEG = W // SEG; NF = SEG // 2 + 1; NCH = 7


In [ ]:
def nt_xent(a, b, tau=TFC_TAU):
    a, b = F.normalize(a, dim=-1), F.normalize(b, dim=-1); N = len(a); z = torch.cat([a, b]); sim = z @ z.T / tau
    sim = sim.masked_fill(torch.eye(2 * N, dtype=torch.bool, device=z.device), float("-inf")); i = torch.arange(2 * N, device=z.device)
    return -F.log_softmax(sim, 1)[i, (i + N) % (2 * N)].mean()

def tfc_loss(ht, hta, hf, hfa, zt, zta, zf, zfa):
    s = nt_xent(zt, zf); terms = [torch.clamp(s - nt_xent(a, b) + TFC_DELTA, min=0) for a, b in ((zt, zfa), (zta, zf), (zta, zfa))]
    return TFC_LAM * (nt_xent(ht, hta) + nt_xent(hf, hfa)) + (1 - TFC_LAM) * sum(terms)

def to_mag(x): return torch.fft.rfft(x.reshape(x.shape[0], NSEG, SEG, x.shape[2]), dim=2).abs()          # (B,NSEG,NF,C)
def freq_aug(mag, alpha=0.5):
    B, S, Fb, C = mag.shape; idx = torch.randint(0, Fb, (B, S, 1, C), device=mag.device); cur = mag.gather(2, idx); mx = mag.amax(2, keepdim=True)
    add = torch.rand(B, 1, 1, 1, device=mag.device) < 0.5
    return mag.scatter(2, idx, torch.where(add, torch.maximum(cur, alpha * mx), torch.zeros_like(cur)))
def time_aug(x):
    B = len(x); op = torch.randint(0, 3, (B, 1, 1), device=x.device)
    sh = torch.stack([torch.roll(x[i], int(np.random.randint(-5, 6)), 0) for i in range(B)])
    return torch.where(op == 0, x + 0.05 * torch.randn_like(x), torch.where(op == 1, x * (1 + 0.1 * torch.randn(B, 1, x.shape[2], device=x.device)), sh))
def freq_in(mag, mu, sd): return ((torch.log1p(mag) - mu) / sd).reshape(mag.shape[0], NSEG, NF * NCH).permute(0, 2, 1).contiguous()

class Cap:
    def __init__(self, m): self.x = None; self.h = m.classifier.register_forward_pre_hook(lambda mod, i: setattr(self, "x", i[0]))

def tfc_pretrain(ent, seed):
    set_seed(seed); rng = np.random.RandomState(seed + 17); dT = DEPTH; dF = DEPTH // 2
    et = VNet(NCH, NC, dT, 0.1).to(dev); ef = VNet(NF * NCH, NC, dF, 0.1).to(dev)
    pt = nn.Sequential(nn.Linear(dT, 256), nn.ReLU(), nn.Linear(256, 128)).to(dev); pf = nn.Sequential(nn.Linear(dF, 256), nn.ReLU(), nn.Linear(256, 128)).to(dev)
    ct, cf = Cap(et), Cap(ef); params = [p for m in (et, ef, pt, pf) for p in m.parameters()]
    opt = optim.Adam(params, lr=3e-4, weight_decay=5e-4)
    Xn, _ = draw_windows(ent, 1000, rng); mag = to_mag(torch.from_numpy(Xn).float()); lm = torch.log1p(mag)
    mu, sd = lm.mean((0, 1), keepdim=True).to(dev), (lm.std((0, 1), keepdim=True) + 1e-6).to(dev)
    def emb(enc, cap, proj, x, k, k_mult):
        if k: x = torch.roll(x, k * k_mult, 2)
        enc(x); h = F.avg_pool1d(cap.x, SEG if k_mult == SEG else 1, SEG if k_mult == SEG else 1).permute(0, 2, 1); z = proj(h)
        if k: h, z = torch.roll(h, -k, 1), torch.roll(z, -k, 1)
        return h.reshape(-1, h.shape[-1]), z.reshape(-1, z.shape[-1])
    BLOCK, PAT_B, best, bad, hist, blk, best_sd = 100, 8, 1e9, 0, [], [], None
    for step in range(PRE_STEPS):
        for m in (et, ef, pt, pf): m.train()
        Xb, _ = draw_windows(ent, PRE_BS, rng); x = torch.from_numpy(Xb).float().to(dev)
        xa = time_aug(x); mg = to_mag(x); mga = freq_aug(mg)
        xt, xta = x.permute(0, 2, 1), xa.permute(0, 2, 1); xf, xfa = freq_in(mg, mu, sd), freq_in(mga, mu, sd); ks = np.random.randint(0, NSEG, 4)
        ht, zt = emb(et, ct, pt, xt, int(ks[0]), SEG); hta, zta = emb(et, ct, pt, xta, int(ks[1]), SEG)
        hf, zf = emb(ef, cf, pf, xf, int(ks[2]), 1); hfa, zfa = emb(ef, cf, pf, xfa, int(ks[3]), 1)
        loss = tfc_loss(ht, hta, hf, hfa, zt, zta, zf, zfa); opt.zero_grad(); loss.backward(); opt.step(); blk.append(loss.item())
        if (step + 1) % BLOCK == 0:
            mval = float(np.mean(blk)); hist.append(mval); blk = []
            print(f"TF-C pre-train step {step+1}/{PRE_STEPS}  loss {mval:.4f}")
            if mval < best - 1e-3: best, bad, best_sd = mval, 0, {k: v.detach().cpu().clone() for k, v in et.state_dict().items()}
            else:
                bad += 1
                if bad >= PAT_B: break
    if best_sd is None: best_sd = {k: v.detach().cpu().clone() for k, v in et.state_dict().items()}
    ct.h.remove(); cf.h.remove()
    return {k: v for k, v in best_sd.items() if not k.startswith("classifier")}, dict(pre_steps=len(hist) * BLOCK, pre_best_loss=best, pre_pool_windows_B_share=float(np.mean([s == "B" for s, _ in ent])))


In [ ]:
def run_c3(pool, ctx):
    fi, seed = ctx["fi"], ctx["seed"]
    rng = np.random.RandomState(seed * 11 + fi); ent = make_pool(pool) + [("L", n) for n in ctx["names_train"]]
    sd, pinfo = tfc_pretrain(ent, seed * 100 + fi)
    m = train_supervised(ctx["names_train"], ctx["Xv"], ctx["Yv"], seed * 100 + fi + 1, init_state=sd); save_state(m, "C3", fi, seed, pool)
    meta = dict(arm=f"C3|{pool}|natural|pf1.0", cond="C3", pool=pool, sampling="natural", pool_frac=1.0, exp="pool", fold=fi + 1,
                test_ind=ctx["test_ind"], seed=seed, frac=1.0, round=1, kind="c3", **pinfo, **m.info)
    row = row_of(meta, m, ctx); print(f"C3 {pool}: test macro {row['test_macro']:.3f} | test target {row['test_target']:.3f}")
    return [row], [], [], []


In [ ]:
run_stage("C3", run_c3, POOLS)
print("C3 units finished:", len(list((OUT / "units_C3").glob("*.pkl"))))
